# Model Deployment with Joblib

Hướng dẫn hoàn chỉnh để save, load và deploy machine learning models cho flight delay prediction.

## Problem Statement

- Sau khi huấn luyện model, ta cần lưu nó để dùng lại trong sản xuất.
- Serialization: chuyển đổi Python objects thành bytes để lưu trữ.
- Joblib: thư viện hiệu quả cho sklearn models và numpy arrays.
- Workflow: Train → Save → Load → Predict (trong Streamlit hoặc ứng dụng khác).

## What is Serialization?

Serialization là quá trình chuyển đổi Python objects (mô hình, dữ liệu) thành dạng có thể lưu trữ (bytes/file) hoặc truyền qua mạng.

### Tại sao cần serialization?

1. **Persistence**: Lưu lại trạng thái của model sau khi huấn luyện.
2. **Reusability**: Load lại model mà không cần huấn luyện lại.
3. **Sharing**: Gửi model qua mạng hoặc giữa các máy.
4. **Production**: Deploy model vào production environment.

## Joblib vs Pickle

| Đặc điểm | Joblib | Pickle |
|---------|--------|--------|
| **Compression** | Có (tiết kiệm bộ nhớ) | Không |
| **Speed** | Nhanh cho numpy arrays | Chậm hơn |
| **Size** | Nhỏ hơn | Lớn hơn |
| **Sklearn** | Tối ưu cho sklearn | Chậm |
| **Compatibility** | Python 2 & 3 | Python 2 & 3 |

**Kết luận**: Joblib tốt hơn cho sklearn models và numpy arrays.

## Importing Required Libraries

In [ ]:
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

print('All libraries imported successfully')

## Helper Functions for Model Serialization

Các hàm này nằm trong `app/model_utils.py` nhưng được giải thích ở đây.

In [ ]:
def save_model(model, feature_names, scaler=None, model_name="model", model_dir=Path("../models")):
    """
    Save a trained model with metadata using joblib.
    
    Args:
        model: Trained sklearn model
        feature_names: List of feature names
        scaler: Optional StandardScaler (for Linear Regression)
        model_name: Name for the model file
        model_dir: Directory to save
    
    Returns:
        Path to saved file
    """
    model_dir = Path(model_dir)
    model_dir.mkdir(parents=True, exist_ok=True)
    
    # Package model with metadata
    model_package = {
        "model": model,
        "feature_names": feature_names,
        "scaler": scaler,
        "model_type": model.__class__.__name__,
    }
    
    # Save with compression
    model_file = model_dir / f"{model_name}.joblib"
    joblib.dump(model_package, model_file, compress=3)
    
    print(f"✓ Model saved to {model_file}")
    print(f"  Model type: {model.__class__.__name__}")
    print(f"  Features: {len(feature_names)}")
    if scaler:
        print(f"  Scaler: {scaler.__class__.__name__}")
    
    return model_file

def load_model(model_path):
    """
    Load a model and metadata from joblib file.
    
    Args:
        model_path: Path to the saved model
    
    Returns:
        Dict with model, features, scaler, model_type
    """
    model_path = Path(model_path)
    
    if not model_path.exists():
        raise FileNotFoundError(f"Model file not found: {model_path}")
    
    model_package = joblib.load(model_path)
    
    print(f"✓ Model loaded from {model_path}")
    print(f"  Model type: {model_package['model_type']}")
    print(f"  Features: {len(model_package['feature_names'])}")
    
    return model_package

print('Helper functions defined')

## Example: Train and Save Linear Regression Model

Giả sử ta đã huấn luyện model, bây giờ save nó.

In [ ]:
# Load sample data
data_path = Path('../data/processed_flight_data.csv')
if not data_path.exists():
    print(f"Warning: {data_path} not found. Creating sample data for demo.")
    # Create sample data for demonstration
    np.random.seed(42)
    sample_size = 100
    X_sample = np.random.randn(sample_size, 10)
    y_sample = np.random.randn(sample_size) * 10
else:
    df = pd.read_csv(data_path)
    selected_features = ['DepDelay', 'CRSElapsedTime', 'Distance', 'DepHour', 
                        'IsWeekend', 'IsRushHour', 'Origin_Freq', 'Dest_Freq', 
                        'Month', 'DayOfWeek']
    target_column = 'ArrDelay'
    model_df = df[selected_features + [target_column]].dropna()
    X_sample = model_df[selected_features].values
    y_sample = model_df[target_column].values

print(f'Sample data shape: {X_sample.shape}')

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X_sample, y_sample, test_size=0.25, random_state=42)

# Scale for Linear Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train model
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

# Evaluate
y_pred = lr_model.predict(X_test_scaled)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f'Linear Regression trained')
print(f'  RMSE: {rmse:.3f}')
print(f'  R2: {r2:.4f}')

## Save Model with Joblib

In [ ]:
# Define feature names
feature_names = ['DepDelay', 'CRSElapsedTime', 'Distance', 'DepHour', 
                 'IsWeekend', 'IsRushHour', 'Origin_Freq', 'Dest_Freq', 
                 'Month', 'DayOfWeek']

# Save Linear Regression model
model_dir = Path('../models')
lr_model_path = save_model(
    model=lr_model,
    feature_names=feature_names,
    scaler=scaler,
    model_name="linear_regression_arrdelay",
    model_dir=model_dir
)

## Load Model and Make Predictions

In [ ]:
# Load the model
loaded_model_package = load_model(lr_model_path)

# Extract components
loaded_model = loaded_model_package['model']
loaded_scaler = loaded_model_package['scaler']
loaded_features = loaded_model_package['feature_names']

print(f'\nLoaded successfully!')
print(f'Features: {loaded_features}')

In [ ]:
# Make predictions with loaded model
X_test_loaded_scaled = loaded_scaler.transform(X_test)
y_pred_loaded = loaded_model.predict(X_test_loaded_scaled)

# Verify predictions match
rmse_loaded = np.sqrt(mean_squared_error(y_test, y_pred_loaded))
r2_loaded = r2_score(y_test, y_pred_loaded)

print(f'Loaded model predictions:')
print(f'  RMSE: {rmse_loaded:.3f}')
print(f'  R2: {r2_loaded:.4f}')
print(f'\n✓ Predictions match original model: {np.allclose(y_pred, y_pred_loaded)}')

## Production Workflow

### 1. Training Phase (Notebook)
```
1. Load data
2. Train model
3. Evaluate on test set
4. Save model with save_model()
```

### 2. Deployment Phase (Streamlit App)
```
1. Load model with load_model()
2. Get user input
3. Apply scaler if needed
4. Make prediction
5. Display result
```

### 3. Run Streamlit App
```bash
cd app/
streamlit run streamlit_app.py
```

## Key Advantages of This Approach

1. **Modularity**: Model training riêng, deployment riêng.
2. **Efficiency**: Không cần retraining, chỉ load và predict.
3. **Reproducibility**: Lưu exact model state + metadata.
4. **Scalability**: Dễ deploy lên multiple servers.
5. **Interoperability**: Joblib files dễ share giữa các máy.

## Best Practices

1. **Lưu metadata**: Feature names, scaler, model type.
2. **Version control**: Lưu model versions (v1, v2, v3).
3. **Test loading**: Đảm bảo model load đúng trước deployment.
4. **Monitor predictions**: Theo dõi prediction drift trong production.
5. **Update regularly**: Retrain model periodically với dữ liệu mới.

## Files Created

1. **app/model_utils.py**: Helper functions `save_model()` và `load_model()`
2. **app/streamlit_app.py**: Streamlit application cho prediction
3. **models/*.joblib**: Saved trained models

## Summary

Serialization và joblib cho phép:
- Save model sau huấn luyện
- Load model cho inference
- Deploy vào production dễ dàng
- Maintain reproducibility
- Scale predictions

Workflow này là standard trong machine learning production.